In [13]:
import zarr
import dask.array as da
from ome_zarr.io import parse_url
from ome_zarr.writer import write_image
from ome_zarr.scale import Scaler
from tifffile import imread
from magicgui.tqdm import tqdm
from pathlib import Path
from natsort import natsorted
import numpy as np

In [2]:
def get_current_tz(file_path):
    """Get current time point and z slice from file name.

    Parameters
    ----------
    file_path : Path
        A Path object from pathlib. It expects a file name with '_t' and '_z' patterns.

    Returns
    -------
    current_t, current_z : Tuple(int, int)
        Current time point and z slice.
    """
    import re
    pattern_t = '_t(\\d+)'
    pattern_z = '_z(\\d+)'
    current_t, current_z = None, None
    file_name = file_path.stem
    matches_z = re.search(pattern_z, file_name)
    if matches_z is not None:
        current_z = int(matches_z.group(1))  # .zfill(2)
    matches_t = re.search(pattern_t, file_name)
    if matches_t is not None:
        current_t = int(matches_t.group(1))
    return current_t, current_z

In [3]:
def get_structured_list_of_paths(file_paths, file_extension):
    """Get structured list of paths.

    Parameters
    ----------
    file_paths : List of paths
        A list of Path objects from pathlib.
    file_extension : str
        A file extension, like '.tif' or '.ptu'.

    Returns
    -------
    t_path_list : List of lists of paths
        A list of lists of Path objects from pathlib. The first list is the time points, the second list is the z slices.
    """
    from natsort import natsorted
    t_path_list = []
    z_path_list = []
    file_paths = natsorted(file_paths)
    previous_t = 1
    for file_path in file_paths:
        if file_path.suffix == file_extension:
            current_t, current_z = get_current_tz(file_path)
            if current_t is not None:
                if current_t > previous_t:
                    t_path_list.append(z_path_list)
                    z_path_list = []
                    previous_t = current_t
                z_path_list.append(file_path)
    # If no timepoints, z_path_list is file_paths
    if current_t is None:
        z_path_list = file_paths
    # Append last timepoint
    t_path_list.append(z_path_list)
    return t_path_list

In [4]:
folder_path = Path(r"D:\Datasets\FLIM\Lifetime Unmixing Project\TMR31_3_sptw\TMR31_3_tif")
file_extension = '.tif'
# Get all file path with specified file extension
file_paths = natsorted([file_path for file_path in folder_path.iterdir(
) if file_path.suffix == file_extension])

In [5]:
list_of_time_point_paths = get_structured_list_of_paths(
        file_paths, file_extension)

In [6]:
list_of_time_point_paths

[[WindowsPath('D:/Datasets/FLIM/Lifetime Unmixing Project/TMR31_3_sptw/TMR31_3_tif/TMR31_3_t1_z1.tif'),
  WindowsPath('D:/Datasets/FLIM/Lifetime Unmixing Project/TMR31_3_sptw/TMR31_3_tif/TMR31_3_t1_z2.tif'),
  WindowsPath('D:/Datasets/FLIM/Lifetime Unmixing Project/TMR31_3_sptw/TMR31_3_tif/TMR31_3_t1_z3.tif'),
  WindowsPath('D:/Datasets/FLIM/Lifetime Unmixing Project/TMR31_3_sptw/TMR31_3_tif/TMR31_3_t1_z4.tif'),
  WindowsPath('D:/Datasets/FLIM/Lifetime Unmixing Project/TMR31_3_sptw/TMR31_3_tif/TMR31_3_t1_z5.tif'),
  WindowsPath('D:/Datasets/FLIM/Lifetime Unmixing Project/TMR31_3_sptw/TMR31_3_tif/TMR31_3_t1_z6.tif'),
  WindowsPath('D:/Datasets/FLIM/Lifetime Unmixing Project/TMR31_3_sptw/TMR31_3_tif/TMR31_3_t1_z7.tif'),
  WindowsPath('D:/Datasets/FLIM/Lifetime Unmixing Project/TMR31_3_sptw/TMR31_3_tif/TMR31_3_t1_z8.tif'),
  WindowsPath('D:/Datasets/FLIM/Lifetime Unmixing Project/TMR31_3_sptw/TMR31_3_tif/TMR31_3_t1_z9.tif'),
  WindowsPath('D:/Datasets/FLIM/Lifetime Unmixing Project/TMR31_

From documentation of `write_image` from ome_zarr: 

Image array MUST be up to 5-dimensional with dimensions ordered (t, c, z, y, x). Image can be a numpy or dask Array.

In [7]:
# Define the path and metadata
output_path = folder_path / (folder_path.stem + '.zarr')
# stack_shape = (5, 1000, 3, 2048, 2048)  # example shape (t, z, c, y, x)
# stack_shape = (2, 267, 19, 39, 256, 256)  # example shape (c, other, t, z, y, x)
stack_shape = (19, 267, 39, 256, 256)  # example shape (t, c, z, y, x), putting utime as channels and storing just one channel
image_dtype = 'float32'

In [8]:

# Metadata for OME-ZARR (example metadata)
metadata = {
    "name": "dataset_name",
    "axes": ["t", "c", "z", "y", "x"],
    "units": ["s", "ps", "um", "um", "um"],
    "physical_sizes": [1, 0.969, 1, 0.271358549, 0.271358549],
}

In [9]:
output_path

WindowsPath('D:/Datasets/FLIM/Lifetime Unmixing Project/TMR31_3_sptw/TMR31_3_tif/TMR31_3_tif.zarr')

In [10]:
store = zarr.DirectoryStore(output_path)
root = zarr.group(store=store, overwrite=True)

In [11]:
scaler = Scaler(copy_metadata=True, downscale=2, in_place=False, labeled=False, max_layer=4, method='nearest')

In [14]:
# Preparing an empty array for initialization
empty_array = np.zeros(stack_shape, dtype=image_dtype)

The step below consumes all my RAM

In [17]:
# Write the image with the scaler for downsampling
write_image(image=empty_array, group=root, scaler=scaler, compute=True, **metadata)

KeyboardInterrupt: 

In [ ]:
# Using dask to load and rechunk data
dask_array = da.from_zarr(output_path)
dask_array = dask_array.rechunk(chunks={1: -1})  # rechunk micro-time axis
da.to_zarr(dask_array, output_path, overwrite=True)

In [17]:

# Fill OME-ZARR array with data
zarr_array = zarr.open(output_path, mode='r+')
for z_paths, i in zip(tqdm(list_of_time_point_paths, label='time_points'), range(len(list_of_time_point_paths))):
    for path, j in zip(tqdm(z_paths, label='z-slices'), range(len(z_paths))):
        data, _ = imread(path)
        if len(data.shape) == 3:
            zarr_array[0, :data.shape[0], i, j, :data.shape[1], :data.shape[2]] = data
        else:
            zarr_array[:data.shape[0], :data.shape[1], i, j, :data.shape[2], :data.shape[3]] = data

print('Done')


TypeError: write_image() missing 1 required positional argument: 'image'